In [19]:
import requests
import pandas as pd
from datetime import datetime
from google.transit import gtfs_realtime_pb2


URL = "https://realtime.gtfs.de/realtime-free.pb"

response = requests.get(URL, timeout=30)


In [20]:
def parse_gtfs_feed(response):
    """
    Parse a GTFS-RT response into a FeedMessage.

    Parameters
    ----------
    response : requests.Response
        HTTP response containing the serialized GTFS-RT feed.

    Returns
    -------
    FeedMessage
        Parsed GTFS-RT feed.
    """
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)

    print(f"Number of entities: {len(feed.entity)}")

    return feed

feed = parse_gtfs_feed(response)


Number of entities: 119287


In [21]:
for entity in feed.entity[:10]:
    print(entity)

id: "339505tu"
trip_update {
  trip {
    trip_id: "339505"
    start_date: "20260905"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 0
    departure {
      delay: 76
      time: 1788607276
    }
    stop_id: "75092"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 1
    arrival {
      delay: -64
      time: 1788607856
    }
    departure {
      delay: 0
      time: 1788607920
    }
    stop_id: "677532"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 2
    arrival {
      delay: 0
      time: 1788608400
    }
    departure {
      delay: 0
      time: 1788608400
    }
    stop_id: "216988"
    schedule_relationship: SCHEDULED
  }
}

id: "933146tu"
trip_update {
  trip {
    trip_id: "933146"
    start_date: "20260905"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 0
    departure {
      delay: 0
      time: 1788606180
    }
    stop_id: "251615"
    

In [22]:
stops_df = pd.read_csv("../data/mvv_stops.csv", delimiter=";")


In [23]:
def create_stop_name_mapping(stops_df):
    """
    Create a mapping from MVV stop IDs to stop names.

    Parameters
    ----------
    stops_df : pandas.DataFrame
        DataFrame containing the MVV stop data. It must contain
        the columns "HstNummer" and "Name ohne Ort".

    Returns
    -------
    dict
        Dictionary mapping stop IDs to stop names.
    """
    stops_df["HstNummer"] = stops_df["HstNummer"].astype(str)

    stop_names = (
        stops_df
        .set_index("HstNummer")["Name ohne Ort"]
        .to_dict()
    )

    return stop_names


In [24]:
def parse_trip_updates(feed, stop_names, trip_lines):
    """
    Parse GTFS-RT trip updates into a pandas DataFrame.

    Parameters
    ----------
    feed : FeedMessage
        Parsed GTFS-RT feed containing trip updates.
    stop_names : dict
        Mapping from stop IDs to stop names.
    trip_lines : dict
        Mapping from trip IDs to line names.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing trip, line, stop, arrival,
        departure, and delay information.
    """
    rows = []

    for entity in feed.entity:
        if not entity.HasField("trip_update"):
            continue

        trip = entity.trip_update.trip

        # Get line for this trip
        line = trip_lines.get(str(trip.trip_id))

        # Ignore trips that are not part of the selected agencies
        if line is None:
            continue

        for stop in entity.trip_update.stop_time_update:

            row = {
                "trip_id": trip.trip_id,
                "start_date": trip.start_date,
                "line": line,
                "stop_id": str(stop.stop_id),
                "stop_name": stop_names.get(str(stop.stop_id)),
                "stop_sequence": stop.stop_sequence,
            }

            if stop.HasField("departure"):
                row["departure_time"] = datetime.fromtimestamp(
                    stop.departure.time
                )
                row["departure_delay"] = stop.departure.delay

            if stop.HasField("arrival"):
                row["arrival_time"] = datetime.fromtimestamp(
                    stop.arrival.time
                )
                row["arrival_delay"] = stop.arrival.delay

            rows.append(row)

    return pd.DataFrame(rows)

In [25]:
def preprocess_gtfs(data_dir, munich_agencies):
    """
    Preprocess GTFS static data for selected agencies.

    Returns
    -------
    trip_lines : dict
        Mapping from trip_id to line name.

    stop_names : dict
        Mapping from stop_id to stop name.
    """

    routes_df = pd.read_csv(f"{data_dir}/routes.txt")
    trips_df = pd.read_csv(f"{data_dir}/trips.txt")
    stops_df = pd.read_csv(f"{data_dir}/stops.txt")

    routes_df["route_id"] = routes_df["route_id"].astype(str)
    routes_df["agency_id"] = routes_df["agency_id"].astype(str)

    trips_df["trip_id"] = trips_df["trip_id"].astype(str)
    trips_df["route_id"] = trips_df["route_id"].astype(str)

    stops_df["stop_id"] = stops_df["stop_id"].astype(str)

    munich_routes = routes_df[
        routes_df["agency_id"].isin(munich_agencies)
    ]

    route_lines = (
        munich_routes
        .set_index("route_id")["route_short_name"]
        .to_dict()
    )

    munich_trips = trips_df[
        trips_df["route_id"].isin(route_lines)
    ]

    trip_lines = (
        munich_trips
        .set_index("trip_id")["route_id"]
        .map(route_lines)
        .to_dict()
    )

    stop_names = (
        stops_df
        .set_index("stop_id")["stop_name"]
        .to_dict()
    )

    return trip_lines, stop_names


In [26]:
munich_agencies = ["100", "191", "364"]

trip_lines, stop_names = preprocess_gtfs(
    "../data",
    munich_agencies
)

In [27]:
df = parse_trip_updates(
    feed,
    stop_names,
    trip_lines
)

df.head(100)

,trip_id,start_date,line,stop_id,stop_name,stop_sequence,departure_time,departure_delay,arrival_time,arrival_delay
0,1012277,20260905,E64,593558,"Dt-Diestelbruch, Alter Krug",15,2026-09-05 13:08:43,-257.0,2026-09-05 13:08:43,-257.0
1,1012277,20260905,E64,307495,"Landshut, Campingplatz",16,2026-09-05 13:09:31,-239.0,2026-09-05 13:09:26,-244.0
2,1012277,20260905,E64,363854,Höne(Dinklage) Höner Ring,17,2026-09-05 13:11:05,-235.0,2026-09-05 13:11:04,-236.0
3,1012277,20260905,E64,54070,Kreuzbruchhof,18,2026-09-05 13:13:56,-184.0,2026-09-05 13:13:21,-219.0
4,1012277,20260905,E64,582647,Niederkrüchten Elmpt Siedlung,19,2026-09-05 13:15:08,-172.0,2026-09-05 13:15:03,-177.0
...,...,...,...,...,...,...,...,...,...,...
95,693569,20260905,17,611632,"Langenh., Ehem. Bahnhof",16,2026-09-05 15:29:00,0.0,2026-09-05 15:29:00,0.0
96,693569,20260905,17,65768,None,17,2026-09-05 15:31:00,0.0,2026-09-05 15:31:00,0.0
97,693569,20260905,17,554014,"Mötsch, Poststelle",18,2026-09-05 15:33:00,0.0,2026-09-05 15:33:00,0.0
98,693569,20260905,17,355309,Tattendorf,19,2026-09-05 15:34:30,0.0,2026-09-05 15:34:30,0.0


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36058 entries, 0 to 36057
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   trip_id          36058 non-null  object        
 1   start_date       36058 non-null  object        
 2   line             36058 non-null  object        
 3   stop_id          36058 non-null  object        
 4   stop_name        35416 non-null  object        
 5   stop_sequence    36058 non-null  int64         
 6   departure_time   32461 non-null  datetime64[ns]
 7   departure_delay  32461 non-null  float64       
 8   arrival_time     31687 non-null  datetime64[ns]
 9   arrival_delay    31687 non-null  float64       
dtypes: datetime64[ns](2), float64(2), int64(1), object(5)
memory usage: 2.8+ MB


In [29]:
def load_existing_realtime_data(
    parquet_path="../data/mvv_realtime.parquet"
):
    """
    Load existing MVV real-time data from a Parquet file.

    Parameters
    ----------
    parquet_path : str
        Path to the existing Parquet file.

    Returns
    -------
    pandas.DataFrame
        Existing MVV real-time data.
    """

    return pd.read_parquet(parquet_path)

In [30]:
def update_realtime_data(
    existing_df,
    new_df
):
    """
    Add new real-time data and keep the latest
    observation for each trip and stop.

    Parameters
    ----------
    existing_df : pandas.DataFrame
        Previously stored MVV real-time data.

    new_df : pandas.DataFrame
        Newly retrieved MVV real-time data.

    Returns
    -------
    pandas.DataFrame
        Updated MVV real-time data.
    """

    combined_df = pd.concat(
        [
            existing_df,
            new_df
        ],
        ignore_index=True
    )

    combined_df = (
        combined_df
        .drop_duplicates(
            subset=[
                "trip_id",
                "start_date",
                "stop_id"
            ],
            keep="last"
        )
        .reset_index(drop=True)
    )

    return combined_df

In [31]:
def save_realtime_data(
    realtime_df,
    parquet_path="../data/mvv_realtime.parquet"
):
    """
    Save MVV real-time data to a Parquet file.

    Parameters
    ----------
    realtime_df : pandas.DataFrame
        MVV real-time data to save.

    parquet_path : str
        Path where the Parquet file is stored.

    Returns
    -------
    None
        The DataFrame is saved to the specified Parquet file.
    """

    realtime_df.to_parquet(
        parquet_path,
        index=False
    )

In [32]:
existing_df = load_existing_realtime_data()

updated_df = update_realtime_data(
    existing_df,
    df
)

save_realtime_data(
    updated_df
)